# Lekcija 02 - Istraživanje Microsoft Agent Frameworka

**Microsoft Agent Framework (MAF)** je jedinstveni okvir za izgradnju AI agenata. Pruža čistu, sastavljivu arhitekturu sa četiri osnovna gradivna bloka:

- **Klijent** – povezuje se s krajnjom točkom AI modela i upravlja komunikacijom
- **Agent** – obavija klijenta upute i definicije alata
- **Alati** – proširuju sposobnosti agenta prilagođenim funkcijama koje model može pozvati
- **Sesija** – održava povijest razgovora za višekratne interakcije

U ovoj lekciji, izgradit ćemo **agenta za rezervaciju putovanja** koji provjerava dostupnost odredišta koristeći ove koncepte.


## Postavljanje


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Razumijevanje arhitekture Agent Frameworka

Microsoft Agent Framework slijedi slojevitu arhitekturu:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klijent** – `FoundryChatClient` se povezuje s Azure OpenAI implementacijom. Upravljat će autentifikacijom, formatiranjem zahtjeva i parsiranjem odgovora.
2. **Agent** – Kreiran iz klijenta putem `provider.create_agent()`, agent kombinira pristup modelu s uputama (sistemski prompt) i alatima.
3. **Alati** – Python funkcije označene s `@tool` koje agent može pozvati za izvođenje radnji ili dohvat podataka.
4. **Sesija** – Objekt `AgentSession` (kreiran putem `agent.create_session()`) koji pohranjuje povijest razgovora, omogućujući višekratni dijalog gdje se agent sjeća prethodnog konteksta.

Izgradimo svaki sloj korak po korak.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Dodavanje alata s dekoratorom @tool

Alati omogućuju agentima poduzimanje radnji osim generiranja teksta. Dekorator `@tool` pretvara uobičajenu Python funkciju u nešto što agent može pozvati.

Ključne točke:
- Koristite `Annotated[type, "opis"]` kako bi model razumio svaki parametar.
- Docstring postaje opis alata koji model vidi.
- `approval_mode="never_require"` znači da se alat pokreće automatski bez potvrde korisnika.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Kreiranje agenta s alatima

Sada kombiniramo klijenta, upute i alate u jednog agenta. `instructions` služe kao sistemski podsjetnik — one definiraju personu i ponašanje agenta.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Višekratni razgovori sa sesijama

`AgentSession` (kreiran pomoću `agent.create_session()`) prati sve poruke u razgovoru. Prosljeđivanjem iste sesije u svaki poziv `agent.run()`, agent ima pristup cijeloj povijesti razgovora i može se vraćati na ranije poruke.

Prosljeđujemo `tools=[check_destination_availability]` kako bi agent mogao pozvati naš alat za provjeru dostupnosti tijekom svakog koraka razgovora.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Sažetak

U ovoj lekciji ste istražili četiri stupca Microsoft Agent Okvira:

| Koncept | Što ste naučili |
|---------|------------------|
| **Klijent** | `FoundryChatClient` se povezuje s Azure OpenAI koristeći autentifikaciju na temelju vjerodajnica |
| **Agent** | `provider.create_agent()` povezuje model s uputama i imenom |
| **Alati** | Dekorator `@tool` izlaže Python funkcije koje agent može pozivati |
| **Sesija** | `agent.create_session()` održava povijest razgovora kroz više koraka |

Ovi građevni blokovi zajedno stvaraju agente koji mogu voditi prirodne razgovore, pozivati vanjske funkcije i održavati kontekst — temelj za naprednije agente u kasnijim lekcijama.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
